In [3]:
import re
import sys

USE_MAH_FOR_SINGLE = True


CONJUNCTS = {
    "क्ष": "ksh",  "ज्ञ": "gya",  "श्र": "shr",
    "प्र": "pra",  "त्र": "tra",  "स्त": "sta",
    "स्थ": "stha", "स्व": "swa",  "द्व": "dwa",
    "द्य": "dya",  "ह्य": "hya",  "ह्र": "hra",
    "ह्व": "hwa",  "न्ह": "nha",  "म्ह": "mha",
    "ल्ह": "lha",  "न्न": "nna",  "त्त": "tta",
    "क्क": "kka",  "ग्ग": "gga",  "ब्ब": "bba",
    "म्म": "mma",  "न्द": "nda",  "न्त": "nta",
    "ङ्क": "ngka", "ङ्ग": "ngga", "ञ्च": "ncha",
    "ञ्ज": "nja",
}       

CONSONANTS = {
    "क": "k",  "ख": "kh", "ग": "g",  "घ": "gh", "ङ": "ng",
    "च": "ch", "छ": "chh","ज": "j",  "झ": "jh", "ञ": "n",
    "ट": "t",  "ठ": "th", "ड": "d",  "ढ": "dh", "ण": "n",
    "त": "t",  "थ": "th", "द": "d",  "ध": "dh", "न": "n",
    "प": "p",  "फ": "ph", "ब": "b",  "भ": "bh", "म": "m",
    "य": "y",  "र": "r",  "ल": "l",
    "व": "w",
    "श": "sh", "ष": "sh", "स": "s",  "ह": "h",
}

VOWEL_SIGNS = {
    "ा": "a",  "ि": "i",  "ी": "i",  "ु": "u",  "ू": "u",
    "ृ": "ri", "े": "e",  "ै": "ai", "ो": "o",  "ौ": "au",
    "ं": "n",  "ँ": "",   "ः": "h",
    "\u094d": "",
}

INDEPENDENT_VOWELS = {
    "अ": "a",  "आ": "aa", "इ": "i",  "ई": "ii",
    "उ": "u",  "ऊ": "uu", "ऋ": "ri",
    "ए": "e",  "ऐ": "ai", "ओ": "o",  "औ": "au",
    "ऍ": "e",  "ऑ": "o",
}

POSTPOSITION_SPLITS = [
    (r'([a-z]{3,})lai\b',   r'\1 lai'),
    (r'([a-z]{3,})bata\b',  r'\1 bata'),
    (r'([a-z]{3,})sanga\b', r'\1 sanga'),
    (r'([a-z]{5,})ma\b',    r'\1 ma'),
    (r'([a-z]{6,})ko\b',    r'\1 ko'),
]

LOANWORDS = {
    "आर्सनललाई":    "Arsenal lai",
    "आर्सनल":       "Arsenal",
    "एथ्लेटिको":    "Atletico",
    "मड्रिड":       "Madrid",
    "युरोपा":       "Europa",
    "प्रिमियर":     "Premier",
    "लिगको":        "League ko",
    "लिग":          "League",
    "फाइनलमा":      "final ma",
    "फाइनल":        "final",
    "सिजन":         "season",
    "कप":           "Cup",
    "बेंगर":        "Wenger",
    "इमिरेट्स":    "Emirates",
    "रियल":         "Real",
    "फुटबल":        "football",
    "च्याम्पियन्स": "Champions",
    "काठमाडौं":     "Kathmandu",
    "वेबसाइट":       "Website",
    'फाइनलमा': 'final ma',
    'सिजनमा': 'season ma',
    'कपको': 'cup ko',
    'प्रशिक्षकका': 'coach ka',
    'प्रशिक्षकले': 'coach le',
    'बेंगरले': 'Wenger le',
    'इमिरेट्स छोड्ने': 'Emirates chodne',
    'उनले': 'unlay',          # human typing style
    'तीन वटा': 'tin baata',
    'सात वटा': 'sat baata',
    'मात्र': 'matra',          # keep final 'a'
}

DIGITS = {"०": "0", "१": "1", "२": "2", "३": "3", "४": "4",
          "५": "5", "६": "6", "७": "7", "८": "8", "९": "9"}

HALANT        = "\u094d"
PUNCT_MAP     = {"।": ".", "॥": ".", "–": "-", "—": "-", "…": "..."}
WORD_BOUNDARY = set(" ।॥\n\t?!,.-–—)]}")

# ═══════════════════════════════════════════════════════════════════════════
#  CORE TRANSLITERATOR (unchanged)
# ═══════════════════════════════════════════════════════════════════════════

def _is_consonant(c: str) -> bool:         return c in CONSONANTS
def _is_vowel_sign(c: str) -> bool:        return c in VOWEL_SIGNS
def _is_independent_vowel(c: str) -> bool: return c in INDEPENDENT_VOWELS
def _is_word_boundary(c: str) -> bool:     return c in WORD_BOUNDARY

def transliterate(text: str) -> str:
    result = []
    chars = list(text)
    n = len(chars)
    i = 0

    while i < n:
        # Loanword
        matched_loan = None
        for loan_len in range(min(10, n - i), 0, -1):
            chunk = "".join(chars[i:i + loan_len])
            if chunk in LOANWORDS:
                matched_loan = (chunk, loan_len)
                break
        if matched_loan:
            result.append(LOANWORDS[matched_loan[0]])
            i += matched_loan[1]
            continue

        ch = chars[i]

        if ch in DIGITS:
            result.append(DIGITS[ch])
            i += 1
            continue

        if ch in PUNCT_MAP:
            result.append(PUNCT_MAP[ch])
            i += 1
            continue

        if _is_independent_vowel(ch):
            result.append(INDEPENDENT_VOWELS[ch])
            i += 1
            continue

        # Conjunct
        conjunct_found = False
        for clen in (3, 2):
            chunk = "".join(chars[i:i + clen])
            if chunk in CONJUNCTS:
                next_i = i + clen
                rom = CONJUNCTS[chunk]
                if next_i < n and _is_vowel_sign(chars[next_i]):
                    vsign = chars[next_i]
                    result.append(rom if vsign == HALANT else rom + VOWEL_SIGNS[vsign])
                    i = next_i + 1
                else:
                    is_final = (next_i >= n or _is_word_boundary(chars[next_i]))
                    result.append(rom if is_final else rom + "a")
                    i = next_i
                conjunct_found = True
                break
        if conjunct_found:
            continue

        # Regular consonant
        if _is_consonant(ch):
            rom = CONSONANTS[ch]
            next_i = i + 1
            if next_i < n:
                nch = chars[next_i]
                if nch == HALANT:
                    result.append(rom)
                    i = next_i + 1
                    continue
                if _is_vowel_sign(nch):
                    result.append(rom + VOWEL_SIGNS[nch])
                    i = next_i + 1
                    continue
                if (_is_word_boundary(nch) or _is_consonant(nch) or _is_independent_vowel(nch)):
                    result.append(rom if _is_word_boundary(nch) else rom + "a")
                    i += 1
                    continue
            # End of string or no special case
            result.append(rom)   # no inherent 'a' at word end
            i += 1
            continue

        # Standalone vowel sign
        if _is_vowel_sign(ch):
            if ch != HALANT:
                result.append(VOWEL_SIGNS[ch])
            i += 1
            continue

        # Passthrough
        result.append(ch)
        i += 1

    return "".join(result)

# ═══════════════════════════════════════════════════════════════════════════
#  POST‑PROCESSING (with configurable single‑word forms)
# ═══════════════════════════════════════════════════════════════════════════

def _fix_va_word_initial(text: str) -> str:
    text = re.sub(r'^w', 'b', text)
    text = re.sub(r'(?<=[ \-\u2013\u2014,.(!\?])w', 'b', text)
    return text

def _fix_chha(text: str) -> str:
    return re.sub(r'chh(?=[^a-zA-Z]|$)', 'chha', text)

def _fix_anusvara_m(text: str) -> str:
    return re.sub(r'n([pbm])', lambda m: 'm' + m.group(1), text)

def _fix_tapain(text: str) -> str:
    text = re.sub(r'tapaii+n?laii+', 'tapai lai', text, flags=re.IGNORECASE)
    text = re.sub(r'tapai+n?lai+',   'tapai lai', text, flags=re.IGNORECASE)
    text = re.sub(r'tapaii+n?',      'tapai',     text, flags=re.IGNORECASE)
    return text

def _fix_single_char_words(text: str) -> str:
    """
    Standalone consonants (म, न, त, etc.) -> 'ma'/'mah', 'na'/'nah', etc.
    Controlled by USE_MAH_FOR_SINGLE flag.
    """
    if USE_MAH_FOR_SINGLE:
        singles = {'m': 'mah', 'n': 'nah', 't': 'tah', 'k': 'kah', 'r': 'rah'}
    else:
        singles = {'m': 'ma', 'n': 'na', 't': 'ta', 'k': 'ka', 'r': 'ra'}
    def repl(m): return singles.get(m.group(1), m.group(1))
    return re.sub(r'(?<![a-zA-Z])([mntkr])(?![a-zA-Z])', repl, text)

def _fix_aan_chandrabindu(text: str) -> str:
    return re.sub(r'\bja+n?(dai|nu|ne|chhu)', r'jaan\1', text)

def _fix_namaste(text: str) -> str:
    return re.sub(r'stae\b', 'ste', text)

def _fix_malai(text: str) -> str:
    return re.sub(r'\bmalaii\b', 'malai', text, flags=re.IGNORECASE)

def _fix_double_vowels(text: str) -> str:
    text = re.sub(r'([^aeiou])aa([^aeiou])', r'\1a\2', text)
    text = re.sub(r'([^aeiou])ao\b',         r'\1o',   text)
    return text

def _split_postpositions(text: str) -> str:
    for pattern, repl in POSTPOSITION_SPLITS:
        text = re.sub(pattern, repl, text)
    return text

def _clean_spaces(text: str) -> str:
    text = re.sub(r' +', ' ', text)
    text = re.sub(r' ([.,!?])', r'\1', text)
    return text.strip()

def _capitalize_first(text: str) -> str:
    return (text[0].upper() + text[1:]) if text else text

def _fix_haraau(text: str) -> str:
    return re.sub(r'\bharau', 'haraau', text)

def _post_process(text: str) -> str:
    text = _fix_va_word_initial(text)
    text = _fix_anusvara_m(text)
    text = _fix_chha(text)
    text = _fix_tapain(text)
    text = _fix_haraau(text)
    text = _fix_aan_chandrabindu(text)
    text = _fix_namaste(text)
    text = _fix_malai(text)
    text = _fix_double_vowels(text)
    text = _split_postpositions(text)
    text = _fix_single_char_words(text)   # uses the flag
    text = _clean_spaces(text)
    text = _capitalize_first(text)
    return text

def romanize_nepali(text: str) -> str:
    return _post_process(transliterate(text))


def main():
    input_file = "devnagari.txt"
    output_file = "output.txt"

    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            lines = [line.rstrip('\n') for line in f]
    except FileNotFoundError:
        print(f"Error: Input file '{input_file}' not found.", file=sys.stderr)
        sys.exit(1)

    results = [romanize_nepali(line) if line.strip() else '' for line in lines]

    with open(output_file, 'w', encoding='utf-8') as out:
        for res in results:
            out.write(res + '\n')

    print(f"Romanized text written to '{output_file}'.")

if __name__ == "__main__":
    main()
    

Romanized text written to 'output.txt'.


In [22]:
import sys
import re
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

class NepaliRomanizerOllamaOnly:
    def __init__(self, model_name: str = "mistral-small3.1:latest", temperature: float = 0.0):
        self.model_name = model_name
        self.temperature = temperature
        self.llm = None
        self.chain = None
        self._initialize_model()
    
    def _initialize_model(self):
        """Initialize the Ollama model"""
        try:
            self.llm = ChatOllama(
                model=self.model_name,
                temperature=self.temperature,
                num_predict=1024,
            )
            
            # Optimized prompt for Mistral
            self.prompt = ChatPromptTemplate.from_messages([
                ("system", """You are a Nepali transliteration expert. Convert Devanagari Nepali text to Roman script.

CRITICAL RULES:
- Output ONLY the romanized text. NO greetings, NO "Sure", NO "Here is", NO explanations.
- Start the output IMMEDIATELY with the converted text.
- Use ONLY plain ASCII letters (a-z). NO diacritics.
- Preserve punctuation, spaces, and line breaks exactly.

Romanization examples:
यसबाट → yasbata
मलाई → malai
दुख → dukh
छैन → chaina
राष्ट्र्प्रेमी → rashtrapremi
साथीहरुले → sathiharle
मान्नु → mannu
धन्यवाद → dhanyabad
भविष्यमा → bhavishyama
गरेको → gareko
रहेछौं → rahechhau

Output format: Start directly with the converted text, no prefixes."""),
                ("human", "{devanagari_text}")
            ])
            
            self.chain = self.prompt | self.llm | StrOutputParser()
            print(f"✓ Model '{self.model_name}' loaded successfully", file=sys.stderr)
            
        except Exception as e:
            print(f"✗ Failed to load model: {e}", file=sys.stderr)
            print("  Make sure Ollama is running: 'ollama serve'", file=sys.stderr)
            print(f"  And model is pulled: 'ollama pull {self.model_name}'", file=sys.stderr)
            sys.exit(1)
    
    def _clean_output(self, text: str) -> str:
        """Remove unwanted prefixes and clean output"""
        # Remove common prefixes
        prefixes = [
            "Sure, here is the Romanized version:",
            "Here is the Romanized version:",
            "Here is the romanized text:",
            "Romanized text:",
            "Output:",
            "Result:",
        ]
        
        for prefix in prefixes:
            if text.startswith(prefix):
                text = text[len(prefix):].strip()
        
        # Remove quotes
        text = re.sub(r'^["\']|["\']$', '', text)
        
        # Ensure proper spacing
        text = re.sub(r'\s+', ' ', text)
        
        # Capitalize first letter of first sentence
        if text and text[0].islower():
            text = text[0].upper() + text[1:]
        
        return text.strip()
    
    def romanize(self, text: str) -> str:
        """Convert Devanagari Nepali text to Roman script"""
        if not text or not text.strip():
            return ""
        
        try:
            # Clean input text
            text = text.strip()
            
            # Invoke the model
            result = self.chain.invoke({"devanagari_text": text})
            
            # Clean up the result
            result = result.strip()
            result = self._clean_output(result)
            
            return result
            
        except Exception as e:
            print(f"✗ Error converting text: {e}", file=sys.stderr)
            print(f"  Text: {text[:50]}...", file=sys.stderr)
            return text
    
    def romanize_batch(self, texts: list) -> list:
        """Convert multiple texts to Roman script"""
        results = []
        total = len(texts)
        
        for idx, text in enumerate(texts, 1):
            if idx % 10 == 0:
                print(f"  Processing {idx}/{total}...", file=sys.stderr)
            
            result = self.romanize(text)
            results.append(result)
        
        return results


def main():
    input_file = "devnagari.txt"
    output_file = "output.txt"
    
    # Use Mistral Small 3.1 - best quality for Nepali
    MODEL_NAME = "mistral-small3.1:latest"
    TEMPERATURE = 0.0
    
    print("=" * 60, file=sys.stderr)
    print("NEPALI ROMANIZATION WITH MISTRAL SMALL 3.1", file=sys.stderr)
    print("=" * 60, file=sys.stderr)
    print(f"Model: {MODEL_NAME}", file=sys.stderr)
    print(f"Temperature: {TEMPERATURE}", file=sys.stderr)
    print("", file=sys.stderr)
    
    # Initialize the romanizer
    romanizer = NepaliRomanizerOllamaOnly(
        model_name=MODEL_NAME,
        temperature=TEMPERATURE
    )
    
    # Read input file
    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            lines = [line.rstrip('\n') for line in f]
        print(f"✓ Read {len(lines)} lines from '{input_file}'", file=sys.stderr)
        print("", file=sys.stderr)
    except FileNotFoundError:
        print(f"✗ Error: Input file '{input_file}' not found.", file=sys.stderr)
        sys.exit(1)
    
    # Process lines
    print("Converting to Roman script...", file=sys.stderr)
    results = romanizer.romanize_batch(lines)
    
    # Write output file
    with open(output_file, 'w', encoding='utf-8') as out:
        for res in results:
            out.write(res + '\n')
    
    print("", file=sys.stderr)
    print("=" * 60, file=sys.stderr)
    print(f"✓ Conversion complete!", file=sys.stderr)
    print(f"  Output written to: '{output_file}'", file=sys.stderr)
    print("=" * 60, file=sys.stderr)


if __name__ == "__main__":
    main()

NEPALI ROMANIZATION WITH MISTRAL SMALL 3.1
Model: mistral-small3.1:latest
Temperature: 0.0

✓ Model 'mistral-small3.1:latest' loaded successfully
✓ Read 3 lines from 'devnagari.txt'

Converting to Roman script...

✓ Conversion complete!
  Output written to: 'output.txt'
